# Finetune BERT using TOLSA-M Data

In [1]:
import os
import sys
import torch
import random
import argparse

import numpy as np
import pandas as pd

from datasets import Dataset
from functools import partial
from datetime import datetime

from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

notebook_dir = os.getcwd()

sys.path.append(os.path.join(notebook_dir, '../'))

from metrics import EvaluationMetric
from data_processing import DataProcessing
from data_visualizing import DataVisualizing


In [2]:
pd.set_option('max_colwidth', 800)

## Notebook Configurations

In [3]:
dataset = '../data/combined_datasets/naacl_2026_submission/naacl_2026_submission.csv'
save_path = '../data/classification_results/naacl_2026_submission'
text_column = 'Base Sentence'
label_column = 'Ground Truth'
# model_name = 'bert-base-cased'
model_name = 'bert-large-cased'
seed = 300
val_size = 0.2
test_size = 0.2

max_length = 512 # Tokenization; in paper, can say this is static instead of dynamic (if we set accroding to each batch); need a lot so we can process mf climate
epochs = 1 # Number of full passes through the training dataset.
learning_rate = 2e-5
weight_decay = 0.01 # Regularization applied to model weights during optimization.
train_batch_size = 16 # Number of sentences processed together in each training batch.
eval_batch_size = 32 # Number of sentences processed together during validation or testing.
logging_steps = 25 # Print training loss and related information every N training steps.
finetuned_model_name = 'tolsa-bert'

# Use mixed-precision floating-point computation only when a GPU is available.
use_fp16 = torch.cuda.is_available()

base_data_path = DataProcessing.load_base_data_path(notebook_dir)

print(f"Base data path: {base_data_path}")
print(f"Dataset: {dataset}")
print(f"Model: {model_name}")
print(f"Seed: {seed}")

Base data path: /Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/notebook_experiments/../data
Dataset: ../data/combined_datasets/naacl_2026_submission/naacl_2026_submission.csv
Model: bert-large-cased
Seed: 300


In [4]:
def set_all_seeds(seed):
    """Set random seeds for reproducible splits and BERT fine-tuning."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)

In [5]:
set_all_seeds(seed)

In [6]:
def resolve_path(base_data_path, file_path):  
    """Return an absolute path for either absolute or data-relative input."""  
    if os.path.isabs(file_path):  
        return file_path  
    return os.path.join(base_data_path, file_path)

## Experiment Setup

In [7]:
current_date = datetime.now().strftime("%Y-%m-%d")
dataset_filename = os.path.basename(dataset)
print(f"\nDataset filename: {dataset_filename}")

dataset_base = os.path.splitext(dataset_filename)[0]
print(f"Dataset base: {dataset_base}")

experiment_name = f"{dataset_base}_{current_date}"
print(f"Experiment name: {experiment_name}")

output_dir = os.path.join(
    save_path,
    experiment_name,
    f"seed{seed}",
    "in_domain",
    model_name,
)

print(f"Output directory: {output_dir}")
os.makedirs(output_dir, exist_ok=True)


Dataset filename: naacl_2026_submission.csv
Dataset base: naacl_2026_submission
Experiment name: naacl_2026_submission_2026-09-25
Output directory: ../data/classification_results/naacl_2026_submission/naacl_2026_submission_2026-09-25/seed300/in_domain/bert-large-cased


## Data: Load + Transform Labels

In [8]:
def load_dataset(base_data_path, dataset_path, text_column, label_column):  
    """Load, validate, and clean a sentence-classification dataset."""  
    print("\n" + "=" * 40)
    print("LOAD DATASET")
    print("=" * 40)
  
    data_path = resolve_path(base_data_path, dataset_path)  
    print(f"Dataset path: {data_path}")  
  
    # Uses your custom DataProcessing class to load the file
    df = DataProcessing.load_from_file(data_path, "csv", sep=",")

    df = df.sample(n=40, random_state=140).reset_index(drop=True)
  
    required_columns = [text_column, label_column]  
    missing_columns = [column for column in required_columns if column not in df.columns]  
  
    if missing_columns:  
        raise ValueError(  
            f"Required columns are missing: {missing_columns}\n"  
            f"Available columns: {list(df.columns)}"  
        )  
  
    original_count = len(df)  
  
    # Remove missing text and missing labels.  
    df = df.dropna(subset=[text_column, label_column]).copy()  
  
    # Ensure all text values are strings.  
    df[text_column] = df[text_column].astype(str).str.strip()  
  
    # Remove blank sentences.  
    df = df[df[text_column] != ""].copy()  
  
    # Standardize labels as strings before creating integer label IDs.  
    df[label_column] = df[label_column].astype(str).str.strip()  
  
    print(f"Original rows: {original_count}")  
    print(f"Usable rows:   {len(df)}")  
    print(f"Shape:         {df.shape}")  
  
    print("\nClass distribution:")  
    print(df[label_column].value_counts())  
  
    print(f"\nPreview:\n{df[[text_column, label_column]].head(5)}\n")  
  
    return df.reset_index(drop=True)  


In [9]:
df = load_dataset(
    base_data_path=base_data_path,
    dataset_path=dataset,
    text_column=text_column,
    label_column=label_column,
)


LOAD DATASET
Dataset path: /Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/notebook_experiments/../data/../data/combined_datasets/naacl_2026_submission/naacl_2026_submission.csv
Original rows: 40
Usable rows:   40
Shape:         (40, 3)

Class distribution:
Ground Truth
0    23
1    17
Name: count, dtype: int64

Preview:
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [10]:
df 

,Base Sentence,Ground Truth,Dataset Name
0,"The goal average at Manchester City should stay the same in 2024-08-28, according to Analyst Sophia Rodriguez.",1,synthetic
1,"And I think that trio now with Desmond Bain as well, like there is a world, I think, where they battle for even a one or a two seed if everything clicks.",1,yt
2,Elisa will expand the use of this technology provided the pilot proves successful .,1,financial_phrasebank
3,"Total public spending is close to 45 per cent of GDP, significantly above pre-pandemic norms, while the tax burden is already at a post-war high and projected to rise further.",1,news_api
4,"A major shift in the upper-level pattern is expected, leading to increased precipitation as an amplified trough moves across the U.S. Expect mostly dry conditions on Thursday, with light lake effect rain/snow in the Great Lakes and isolated rain showers in South Florida. A substantial increase in precipitation is anticipated from Friday into early next week; higher elevation snow and lower elevation rain are likely in the Pacific Northwest and the Rockies. The GFS predicts heavier rains for southern Texas, while the ECMWF keeps southern Plains dry. Temperatures will be below normal by 10-20Â°F across the Plains to the Southeast, with lows potentially in the 30s, possibly setting records. Warm conditions will persist in the Northwest with high temperatures challenging daily records. As ...",1,mf_climate
5,"Some of the most recent technology deliveries include refinery technology to Anrak Aluminium , an iron ore pelletizing plant to Tata Steel and iron ore sintering plants to Bhushan Steel .",0,financial_phrasebank
6,Metso estimates the need for personnel reductions to amount to 25 people .,1,financial_phrasebank
7,"A warming trend is expected across most of the continental United States (CONUS) by the end of the week into next weekend, with temperatures trending above average, particularly near the Canadian border and the Gulf Coast. Rainfall will be modest to locally heavy, spreading from the Lower Mississippi Valley northeastward from Thursday to Sunday, alongside repetitive precipitation in the Pacific Northwest due to multiple systems. A low pressure system is predicted to move northeastward towards the Midwest, weakening by day 6, while a second low pressure system is expected to develop off the Mid-Atlantic coast and track northward by day 7. The Northwest Gulf Coast may experience modest rainfall totals, with some areas potentially seeing locally heavy amounts. In the Northwest, snow level...",1,mf_climate
8,Maybe it would have athletes skiing on fake snow down parched slopes.,0,chronicle2050
9,"When Judge Ketanji Brown Jackson’s Supreme Court nomination reaches the Senate floor soon, every Republican who votes against her confirmation will be complicit in the abuse that the Republican members of the Judiciary Committee heaped on her.",0,chronicle2050


In [11]:
def create_label_mapping(df, label_column):
    """Create BERT-compatible integer IDs for exactly two labels."""
    unique_labels = sorted(df[label_column].unique().tolist())

    if len(unique_labels) != 2:
        raise ValueError(
            "This pipeline currently supports binary classification only. "
            f"Found {len(unique_labels)} labels: {unique_labels}"
        )

    label_to_id = {
        label: index
        for index, label in enumerate(unique_labels)
    }

    id_to_label = {
        index: label
        for label, index in label_to_id.items()
    }

    print("\nLabel mapping:")
    for label, label_id in label_to_id.items():
        print(f"  {label_id} -> {label}")

    return label_to_id, id_to_label

In [12]:
label_to_id, id_to_label = create_label_mapping(
    df=df,
    label_column=label_column,
)
label_to_id, id_to_label


Label mapping:
  0 -> 0
  1 -> 1


({'0': 0, '1': 1}, {0: '0', 1: '1'})

In [13]:
num_labels = len(label_to_id)

if num_labels < 2:
    raise ValueError(
        "Sentence classification requires at least two unique labels."
    )

In [14]:
label_id_column = "_label_id"
df[label_id_column] = df[label_column].map(label_to_id).astype(int)

print(f"Usable rows:   {len(df)}")
print(f"Shape:         {df.shape}")  

print("\nClass distribution:")  
print(df[label_id_column].value_counts())  

df[[text_column, label_id_column]].head(5)

Usable rows:   40
Shape:         (40, 4)

Class distribution:
_label_id
0    23
1    17
Name: count, dtype: int64


,Base Sentence,_label_id
0,"The goal average at Manchester City should stay the same in 2024-08-28, according to Analyst Sophia Rodriguez.",1
1,"And I think that trio now with Desmond Bain as well, like there is a world, I think, where they battle for even a one or a two seed if everything clicks.",1
2,Elisa will expand the use of this technology provided the pilot proves successful .,1
3,"Total public spending is close to 45 per cent of GDP, significantly above pre-pandemic norms, while the tax burden is already at a post-war high and projected to rise further.",1
4,"A major shift in the upper-level pattern is expected, leading to increased precipitation as an amplified trough moves across the U.S. Expect mostly dry conditions on Thursday, with light lake effect rain/snow in the Great Lakes and isolated rain showers in South Florida. A substantial increase in precipitation is anticipated from Friday into early next week; higher elevation snow and lower elevation rain are likely in the Pacific Northwest and the Rockies. The GFS predicts heavier rains for southern Texas, while the ECMWF keeps southern Plains dry. Temperatures will be below normal by 10-20Â°F across the Plains to the Southeast, with lows potentially in the 30s, possibly setting records. Warm conditions will persist in the Northwest with high temperatures challenging daily records. As ...",1


## Data Splits: Create and Save Train/Val/Test

In [15]:
def split_dataset(df, label_id_column, test_size, val_size, seed):
    """Create stratified train, validation, and optional test data splits."""
    print("\n" + "=" * 40)
    print("CREATE DATA SPLITS")
    print("=" * 40)

    if not 0 <= test_size < 1:
        raise ValueError("test_size must be at least 0 and less than 1.")

    if not 0 < val_size < 1:
        raise ValueError("val_size must be greater than 0 and less than 1.")

    if test_size > 0:
        train_val_df, test_df = train_test_split(
            df,
            test_size=test_size,
            random_state=seed,
            stratify=df[label_id_column],
        )

        adjusted_val_size = val_size / (1 - test_size)
    else:
        train_val_df = df.copy()
        test_df = None
        adjusted_val_size = val_size

    train_df, val_df = train_test_split(
        train_val_df,
        test_size=adjusted_val_size,
        random_state=seed,
        stratify=train_val_df[label_id_column],
    )

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)

    if test_df is not None:
        test_df = test_df.reset_index(drop=True)

    print(f"Train size: {len(train_df)}")
    print(f"Validation size: {len(val_df)}")
    print(f"Test size: {len(test_df) if test_df is not None else 0}")

    return train_df, val_df, test_df


In [16]:
train_df, val_df, test_df = split_dataset(
    df=df,
    label_id_column=label_id_column,
    test_size=test_size,
    val_size=val_size,
    seed=seed,
)


CREATE DATA SPLITS
Train size: 24
Validation size: 8
Test size: 8


In [17]:
def save_data_splits(train_df, val_df, test_df, output_dir):
    """Save the exact train, validation, and test splits used."""
    split_dir = os.path.join(output_dir, "data_splits")
    os.makedirs(split_dir, exist_ok=True)

    DataProcessing.save_to_file(
        train_df,
        path=split_dir,
        prefix="x_y_train_set",
        save_file_type="csv",
        include_version=False,
    )

    DataProcessing.save_to_file(
        val_df,
        path=split_dir,
        prefix="x_y_val_set",
        save_file_type="csv",
        include_version=False,
    )

    if test_df is not None:
        DataProcessing.save_to_file(
            test_df,
            path=split_dir,
            prefix="x_y_test_set",
            save_file_type="csv",
            include_version=False,
        )

    print(f"✓ Saved data splits to: {split_dir}")

In [18]:
save_data_splits(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    output_dir=output_dir,
)

Saving CSV file to: ../data/classification_results/naacl_2026_submission/naacl_2026_submission_2026-09-25/seed300/in_domain/bert-large-cased/data_splits/x_y_train_set.csv
Saving CSV file to: ../data/classification_results/naacl_2026_submission/naacl_2026_submission_2026-09-25/seed300/in_domain/bert-large-cased/data_splits/x_y_val_set.csv
Saving CSV file to: ../data/classification_results/naacl_2026_submission/naacl_2026_submission_2026-09-25/seed300/in_domain/bert-large-cased/data_splits/x_y_test_set.csv
✓ Saved data splits to: ../data/classification_results/naacl_2026_submission/naacl_2026_submission_2026-09-25/seed300/in_domain/bert-large-cased/data_splits


## HF Preparation Dataset for Tokenization

In [19]:
def dataframe_to_hf_dataset(df, text_column, label_id_column):
    """Convert a pandas DataFrame to a Hugging Face Dataset."""
    hf_df = df[[text_column, label_id_column]].copy()
    hf_df = hf_df.rename(columns={label_id_column: "labels"})

    return Dataset.from_pandas(hf_df, preserve_index=False)

In [20]:
train_dataset = dataframe_to_hf_dataset(
    train_df,
    text_column=text_column,
    label_id_column=label_id_column,
)

if val_df is not None:
    val_dataset = dataframe_to_hf_dataset(
        val_df,
        text_column=text_column,
        label_id_column=label_id_column
    )

if test_df is not None:
    test_dataset = dataframe_to_hf_dataset(
        test_df,
        text_column=text_column,
        label_id_column=label_id_column
    )

## BERT: Tokenize + Finetune

### Tokenize

In [21]:
def tokenize_batch(examples, tokenizer, text_column, max_length):
    """Tokenize a batch of sentences using a pretrained tokenizer.
    Apply truncation and maximum sequence length constraints.
    Return tokenized model inputs (e.g., input IDs and attention masks).
    """
    return tokenizer(
        examples[text_column],
        truncation=True,
        max_length=max_length,
    )

In [22]:
def tokenize_dataset(
    dataset, 
    tokenizer, 
    text_column, 
    max_length, 
    split_name="dataset", 
    original_df=None, 
    label_column=None, 
    show_examples=False
):
    """Tokenize all sentences in a Hugging Face dataset.
    Apply preprocessing and convert text into BERT-compatible inputs.
    Optionally display tokenization examples for inspection.
    """
    # Use functools.partial to pass the extra arguments without nesting
    tokenized_dataset = dataset.map(
        partial(
            tokenize_batch, 
            tokenizer=tokenizer, 
            text_column=text_column, 
            max_length=max_length
        ),
        batched=True,
        remove_columns=[text_column],
    )

    # Print tokenized count
    print(f"✓ Tokenized {split_name} sentences: {len(tokenized_dataset)}")

    # Print mini-example if requested
    if show_examples and original_df is not None and label_column is not None:
        print("\n" + "=" * 40)
        print(f"TOKENIZATION MINI EXAMPLE — 3 {split_name} samples")
        print("=" * 40)

        # Ensure we don't try to print more examples than exist
        num_examples = min(3, len(tokenized_dataset))
        
        for i in range(num_examples):
            example = tokenized_dataset[i]
            original_text = original_df.iloc[i][text_column]
            label = original_df.iloc[i][label_column]

            tokens = tokenizer.convert_ids_to_tokens(example["input_ids"])
            num_real_tokens = sum(example["attention_mask"])
            num_pad_tokens = len(example["attention_mask"]) - num_real_tokens

            print(f"\n--- Example {i + 1} ---")
            print(f"  Original sentence : {original_text}")
            print(f"  Label ID          : {label}")
            print(f"  Tokens            : {tokens[:num_real_tokens]}")
            print(f"  input_ids         : {example['input_ids'][:num_real_tokens]}")
            print(f"  attention_mask    : {example['attention_mask'][:num_real_tokens]} "
                  f"(+ {num_pad_tokens} zeros for padding)")
            print(f"  Total length      : {len(example['input_ids'])} (max_length={max_length})")

    return tokenized_dataset

In [23]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenized_train_dataset = tokenize_dataset(
    dataset=train_dataset,
    tokenizer=tokenizer,
    text_column=text_column,
    max_length=max_length,
    split_name="training",
    original_df=train_df,
    label_column=label_id_column,
    show_examples=True  # This triggers the mini-example printout
)

if val_df is not None and val_dataset:
    tokenized_val_dataset = tokenize_dataset(
        dataset=val_dataset,
        tokenizer=tokenizer,
        text_column=text_column,
        max_length=max_length,
        split_name="validation"
    )
else:
    tokenized_val_dataset = None
    print("✓ Tokenized validation sentences: 0")

if test_df is not None and test_dataset:
    tokenized_test_dataset = tokenize_dataset(
        dataset=test_dataset,
        tokenizer=tokenizer,
        text_column=text_column,
        max_length=max_length,
        split_name="test"
    )
else:
    tokenized_test_dataset = None
    print("✓ Tokenized test sentences:       0")

tokenized_train_dataset, tokenized_val_dataset, tokenized_test_dataset

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/24 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

✓ Tokenized validation sentences: 8


Map:   0%|          | 0/8 [00:00<?, ? examples/s]

✓ Tokenized test sentences: 8


(Dataset({
     features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
     num_rows: 24
 }),
 Dataset({
     features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
     num_rows: 8
 }),
 Dataset({
     features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
     num_rows: 8
 }))

### Finetune

#### Step 1: Load Pretrained BERT

From the D2L tutorial [1]: BERT is pretrained on BookCorpus + English Wikipedia (800M + 2.5B words) using MLM and NSP tasks.

Here we are **not pretraining**. We load pretrained BERT weights and attach a new binary classification head on top.

The warning *"Some weights not initialized"* is expected. It refers to `classifier.weight` and `classifier.bias`, which are randomly initialized and learned during fine-tuning.

From the Hugging Face tutorial: this is called **transfer learning**.

The model configuration includes:

- `num_labels=2`: binary classification
- `label2id`: mapping from original labels to integer IDs
- `id2label`: reverse mapping from integer IDs to original labels

The printed parameter counts show:

- **Total parameters**: all pretrained BERT parameters plus the new classification-head parameters
- **Trainable parameters**: parameters updated during fine-tuning

In [24]:
def load_pretrained_bert_classifier(model_name, label_to_id, id_to_label):
    """Load pretrained BERT and add a randomly initialized binary classifier."""
    print(f"\n[1/5] Loading pretrained BERT checkpoint: {model_name}")
    print("      num_labels  : 2 (binary classification)")
    print(f"      label2id    : {label_to_id}")
    print(f"      id2label    : {id_to_label}")

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        label2id=label_to_id,
        id2label=id_to_label,
    )

    total_params = sum(parameter.numel() for parameter in model.parameters())
    trainable_params = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

    print(f"      Total parameters     : {total_params:,}")
    print(f"      Trainable parameters : {trainable_params:,}")

    return model

In [25]:
model = load_pretrained_bert_classifier(model_name, label_to_id, id_to_label)
model


[1/5] Loading pretrained BERT checkpoint: bert-large-cased
      num_labels  : 2 (binary classification)
      label2id    : {'0': 0, '1': 1}
      id2label    : {0: '0', 1: '1'}


model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-large-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


      Total parameters     : 333,581,314
      Trainable parameters : 333,581,314


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 1024, padding_idx=0)
      (position_embeddings): Embedding(512, 1024)
      (token_type_embeddings): Embedding(2, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-23): 24 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
              (LayerNorm): LayerNorm((1

#### Step 2: Configure Training Arguments

From the Hugging Face tutorial: `TrainingArguments` controls **how** the `Trainer` fine-tunes the model, including the learning rate, batch sizes, epochs, evaluation frequency, checkpoint saving, and logging.

From D2L [1]: BERT uses Adam optimization with weight decay.

| Argument | Meaning |
|---|---|
| `output_dir` | Location for intermediate training checkpoints |
| `learning_rate` | Step size used to update BERT parameters and the new classification head |
| `per_device_train_batch_size` | Number of training sentences processed together per device |
| `per_device_eval_batch_size` | Number of validation sentences processed together per device |
| `num_train_epochs` | Number of complete passes through the training dataset |
| `weight_decay` | Regularization penalty applied to model weights |
| `eval_strategy="epoch"` | Evaluate after every complete pass through the training data |
| `save_strategy="epoch"` | Save a checkpoint after every epoch |
| `logging_strategy="steps"` | Print training information every specified number of training steps |
| `logging_steps` | Number of training steps between log outputs |
| `save_total_limit=1` | Keep only one checkpoint on disk |
| `fp16` | Use mixed-precision computation when a CUDA GPU is available |
| `report_to=[]` | Do not send logs to external tracking services |
| `seed` | Random seed for reproducibility |

In [26]:
def configure_finetuning_arguments(
    output_dir,
    learning_rate,
    train_batch_size,
    eval_batch_size,
    epochs,
    weight_decay,
    logging_steps,
    seed,
):
    """Configure Hugging Face settings for BERT fine-tuning."""
    checkpoint_dir = os.path.join(output_dir, "training_checkpoints")
    use_fp16 = torch.cuda.is_available()

    print("\n[2/5] Configuring TrainingArguments")
    print(f"      Output dir          : {checkpoint_dir}")
    print(f"      Learning rate       : {learning_rate}")
    print(f"      Train batch size    : {train_batch_size}")
    print(f"      Eval batch size     : {eval_batch_size}")
    print(f"      Epochs              : {epochs}")
    print(f"      Weight decay        : {weight_decay}")
    print("      Eval strategy       : epoch")
    print("      Save strategy       : epoch")
    print(f"      Logging steps       : {logging_steps}")
    print("      Save total limit    : 1")
    print(f"      FP16               : {use_fp16}")
    print(f"      Seed                : {seed}")

    training_args = TrainingArguments(
        output_dir=checkpoint_dir,
        learning_rate=learning_rate,
        per_device_train_batch_size=train_batch_size,
        per_device_eval_batch_size=eval_batch_size,
        num_train_epochs=epochs,
        weight_decay=weight_decay,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=logging_steps,
        save_total_limit=1,
        fp16=use_fp16,
        report_to=[],
        seed=seed,
    )

    return training_args

In [27]:
training_args = configure_finetuning_arguments(
    output_dir,
    learning_rate,
    train_batch_size,
    eval_batch_size,
    epochs,
    weight_decay,
    logging_steps,
    seed
)

training_args


[2/5] Configuring TrainingArguments
      Output dir          : ../data/classification_results/naacl_2026_submission/naacl_2026_submission_2026-09-25/seed300/in_domain/bert-large-cased/training_checkpoints
      Learning rate       : 2e-05
      Train batch size    : 16
      Eval batch size     : 32
      Epochs              : 1
      Weight decay        : 0.01
      Eval strategy       : epoch
      Save strategy       : epoch
      Logging steps       : 25
      Save total limit    : 1
      FP16               : False
      Seed                : 300


TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=False,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False

#### Step 3: Build the Hugging Face x BERT Metrics + Trainer

From the Hugging Face tutorial: `Trainer` combines the model, training configuration, tokenized datasets, padding logic, and metric function into one training object.

`DataCollatorWithPadding` dynamically pads every batch to the length of the longest sentence in that specific batch. This is more efficient than padding every sentence in the dataset to `max_length`.

`compute_metrics` tells `Trainer` to call `EvaluationMetric` after each evaluation cycle to calculate:

- Accuracy
- Precision
- Recall
- F1 Score

The `Trainer` receives:

| Input | Purpose |
|---|---|
| `model` | Pretrained BERT plus the binary classification head |
| `args` | `TrainingArguments` configuration |
| `train_dataset` | Tokenized training sentences and labels |
| `eval_dataset` | Tokenized validation sentences and labels |
| `tokenizer` | The tokenizer matched to the BERT checkpoint |
| `data_collator` | Dynamic batch-padding mechanism |
| `compute_metrics` | Function that computes validation classification metrics |

In [28]:
def bert_trainer_metrics(eval_pred):
    """Compute validation metrics from Hugging Face model outputs."""
    logits, y_true = eval_pred
    # logits: raw output scores for each class, shape (N, 2).
    # y_true: ground-truth label IDs from the validation dataset, shape (N,).
    y_pred = np.argmax(logits, axis=-1)

    return EvaluationMetric.custom_evaluation_metrics(
        y_true=y_true,
        y_prediction=y_pred,
    )

In [29]:
def build_bert_trainer(
    model,
    training_args,
    train_dataset,
    val_dataset,
    tokenizer,
):
    """Combine model, datasets, tokenizer, metrics, and configuration."""
    print("\n[3/5] Building Trainer")
    print(f"      Train examples      : {len(train_dataset)}")
    print(f"      Validation examples : {len(val_dataset)}")
    print("      Data collator       : dynamic batch padding")
    print("      Compute metrics     : Accuracy, Precision, Recall, F1 Score")

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=bert_trainer_metrics,
    )

    return trainer

In [30]:
trainer = build_bert_trainer(model,
    training_args,
    tokenized_train_dataset,
    tokenized_val_dataset,
    tokenizer
)

trainer

/var/folders/78/9z0b45fx1xqbwxh8vk97lcfh0000gn/T/ipykernel_63382/2044725375.py:15: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(



[3/5] Building Trainer
      Train examples      : 24
      Validation examples : 8
      Data collator       : dynamic batch padding
      Compute metrics     : Accuracy, Precision, Recall, F1 Score


#### Step 4: Finetune BERT


From D2L [1]: fine-tuning updates **all** pretrained Transformer encoder parameters while simultaneously training the newly added classification head.

The `[CLS]` token representation flows into the classification head. During BERT pretraining, the `[CLS]` representation is used by the NSP classifier because self-attention allows it to encode information from the complete BERT input sequence [1]. For this downstream sentence-classification task, the pretrained model’s sequence-level representation is used to predict the binary class.

During training, the Hugging Face `Trainer` prints information every `logging_steps` steps:

| Training Log | Meaning |
|---|---|
| `loss` | Cross-entropy loss on the current training batch |
| `grad_norm` | Gradient magnitude; very large spikes can indicate instability |
| `learning_rate` | Current learning-rate value; typically decays toward zero |
| `epoch` | Fractional progress through training; e.g., `1.5` means halfway through epoch 2 |

At the end of every epoch, the trainer prints validation metrics:

| Evaluation Log | Meaning |
|---|---|
| `eval_loss` | Cross-entropy loss on validation data |
| `eval_Accuracy` | Proportion of correct validation predictions |
| `eval_Precision` | Proportion of positive predictions that were correct |
| `eval_Recall` | Proportion of actual positive examples correctly found |
| `eval_F1 Score` | Harmonic mean of precision and recall |

In [31]:
def run_bert_finetuning(trainer, logging_steps):
    """Run supervised fine-tuning on the labeled training sentences."""
    print("\n[4/5] Starting fine-tuning...")
    print(f"      Logging frequency   : every {logging_steps} steps")
    print("      Training logs       : loss, grad_norm, learning_rate, epoch")
    print("      Validation logs     : eval_loss, Accuracy, Precision, Recall, F1 Score")

    trainer.train()

    return trainer

In [32]:
print("train_dataset columns:", train_dataset.column_names)
print("tokenized_train_dataset columns:", tokenized_train_dataset.column_names)
print("tokenized train example:", tokenized_train_dataset[0])

train_dataset columns: ['Base Sentence', 'labels']
tokenized_train_dataset columns: ['labels', 'input_ids', 'token_type_ids', 'attention_mask']
tokenized train example: {'labels': 0, 'input_ids': [101, 1332, 5274, 26835, 5108, 3454, 2671, 3160, 787, 188, 3732, 2031, 6294, 5965, 1103, 3279, 1837, 1770, 117, 1451, 3215, 1150, 3667, 1222, 1123, 15468, 1209, 1129, 3254, 1643, 18726, 1107, 1103, 6704, 1115, 1103, 3215, 1484, 1104, 1103, 25932, 2341, 19943, 1174, 1113, 1123, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [33]:
trainer = run_bert_finetuning(trainer, logging_steps)
trainer


[4/5] Starting fine-tuning...
      Logging frequency   : every 25 steps
      Training logs       : loss, grad_norm, learning_rate, epoch
      Validation logs     : eval_loss, Accuracy, Precision, Recall, F1 Score


/Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/.venv_predictions/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1 score
1,No log,0.758376,0.500000,0.000000,0.000000,0.000000


/Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/.venv_predictions/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


#### Step 5: Save the Finetuned Model and Tokenizer

The fine-tuned BERT model and its matching tokenizer are saved in `final_model`.

This saves the model weights, model configuration, classifier head, tokenizer vocabulary, and tokenizer configuration. The saved model can later be loaded for inference without retraining.

In [34]:
def save_finetuned_bert(finetuned_model_name, trainer, tokenizer, output_dir):
    """Save the fine-tuned model weights, configuration, and tokenizer."""
    final_model_dir = os.path.join(output_dir, finetuned_model_name)

    trainer.save_model(final_model_dir)
    tokenizer.save_pretrained(final_model_dir)

    print("\n[5/5] Fine-tuning complete.")
    print(f"      ✓ Saved fine-tuned BERT model to : {final_model_dir}")
    print(f"      ✓ Saved tokenizer to             : {final_model_dir}")

    return final_model_dir

In [35]:
finetuned_bert_saved = save_finetuned_bert(finetuned_model_name, trainer, tokenizer, output_dir)


[5/5] Fine-tuning complete.
      ✓ Saved fine-tuned BERT model to : ../data/classification_results/naacl_2026_submission/naacl_2026_submission_2026-09-25/seed300/in_domain/bert-large-cased/tolsa-bert
      ✓ Saved tokenizer to             : ../data/classification_results/naacl_2026_submission/naacl_2026_submission_2026-09-25/seed300/in_domain/bert-large-cased/tolsa-bert


## Evaluation: Validation + Test Sets

### Helper Functions

In [36]:
def get_finetuned_bert_predictions(trainer, tokenized_dataset):
    """Generate BERT predicted labels and probability vectors."""

    print("\n" + "=" * 40)
    print("BERT PREDICTIONS")
    print("=" * 40)

    prediction_output = trainer.predict(tokenized_dataset)

    logits = prediction_output.predictions
    shifted_logits = logits - np.max(logits, axis=1, keepdims=True)
    exp_logits = np.exp(shifted_logits)
    probabilities = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)
    y_pred = np.argmax(probabilities, axis=1)

    print(f"Number of BERT predictions : {len(y_pred)}")
    print(f"First 5 predicted labels   : {y_pred[:5]}")
    print(f"Probability matrix shape   : {probabilities.shape}")
    print(f"First probability vector   : {probabilities[0]}")

    return y_pred, probabilities


In [37]:
def evaluate_bert_predictions(
    y_true,
    y_pred,
    probabilities,
    split_name,
    seed,
    save_path,
    model_name,
):
    """
    Evaluate BERT predictions and save unified metrics and visualizations.

    Parameters
    ----------
    y_true : np.ndarray
        Ground-truth label IDs.
    y_pred : np.ndarray
        Predicted label IDs.
    probabilities : np.ndarray
        Softmax probability matrix, shape (N, 2).
    split_name : str
        Split identifier, e.g. 'validation' or 'test'.
    seed : int
        Random seed used in the experiment.
    save_path : str
        Directory path to save visualizations and metrics.
    model_name : str
        Name used for saving files and labeling outputs.

    Returns
    -------
    tuple
        (metrics_df, confusion_matrix, eval_report)
    """

    print("\n" + "=" * 40)
    print(f"EVALUATE BERT: {split_name.upper()}")
    print("=" * 40)

    # ============================================================
    # CLASSIFICATION REPORT
    # ============================================================
    eval_report = EvaluationMetric.eval_classification_report(
        y_true,
        y_pred,
    )

    # ============================================================
    # CONFUSION MATRIX
    # ============================================================
    cm, tn, fp, fn, tp = EvaluationMetric.get_confusion_matrix(
        y_true,
        y_pred,
        by_category=True,
    )

    print(f"Confusion Matrix:\n{cm}\n")
    print(f"TN: {tn}, FP: {fp}, FN: {fn}, TP: {tp}\n")

    save_visualizations_path = os.path.join(save_path, "visualizations")
    DataVisualizing.confusion_matrix(
        model_name,
        cm,
        save_visualizations_path,
        include_version=False,
    )
    print(f"✓ Saved confusion matrix: confusion_matrix_{model_name}.png\n")

    # ============================================================
    # AUC SCORES + CURVES
    # ============================================================
    positive_class_probabilities = probabilities[:, 1]

    roc_auc_score = EvaluationMetric.get_roc_auc(
        y_true,
        positive_class_probabilities,
    )
    print(f"ROC-AUC Score: {roc_auc_score:.4f}\n")

    # DataVisualizing.roc_curve(
    #     model_name,
    #     None,
    #     positive_class_probabilities,
    #     y_true,
    #     save_visualizations_path,
    #     include_version=False,
    # )
    # print(f"✓ Saved ROC-CURVE: roc_curve_{model_name}.png\n")

    pr_auc_score = EvaluationMetric.get_pr_auc(
        y_true,
        positive_class_probabilities,
    )
    print(f"PR-AUC Score: {pr_auc_score:.4f}\n")

    # DataVisualizing.pr_curve(
    #     model_name,
    #     None,
    #     positive_class_probabilities,
    #     y_true,
    #     save_visualizations_path,
    #     include_version=False,
    # )
    # print(f"✓ Saved PR-CURVE: pr_curve_{model_name}.png\n")

    # ============================================================
    # BUILD METRICS ROW
    # ============================================================
    metrics_df = pd.DataFrame([{
        "seed":               seed,
        "model":              model_name,
        "split":              split_name,
        # No training phase for inference-only evaluation
        "train_accuracy":     None,
        "val_accuracy":       None,
        "test_accuracy":      eval_report.get("accuracy", None),
        "precision_class_0":  eval_report.get("0", {}).get("precision", None),
        "precision_class_1":  eval_report.get("1", {}).get("precision", None),
        "recall_class_0":     eval_report.get("0", {}).get("recall", None),
        "recall_class_1":     eval_report.get("1", {}).get("recall", None),
        "f1_class_0":         eval_report.get("0", {}).get("f1-score", None),
        "f1_class_1":         eval_report.get("1", {}).get("f1-score", None),
        "tn":                 tn,
        "fp":                 fp,
        "fn":                 fn,
        "tp":                 tp,
        "roc_auc":            roc_auc_score,
        "pr_auc":             pr_auc_score,
    }])

    print("\n" + "=" * 40)
    print(f"METRICS SUMMARY WITH SEED {seed}")
    print("=" * 40)
    print(metrics_df)
    print()

    return metrics_df, cm, eval_report

In [38]:
def create_bert_results_dataframe(source_df, y_pred, probabilities, id_to_label):
    """Attach BERT predictions and probabilities to source data."""

    print("\n" + "=" * 40)
    print("CREATE BERT RESULTS DATAFRAME")
    print("=" * 40)

    results_df = source_df.copy()

    if "Ground Truth" in results_df.columns:
        results_df["Ground Truth"] = results_df["Ground Truth"].astype(int)

    results_df["BERT Predicted Label ID"] = y_pred
    results_df["BERT Predicted Label"] = [id_to_label[pred] for pred in y_pred]
    results_df["BERT Probability Class 0"] = probabilities[:, 0]
    results_df["BERT Probability Class 1"] = probabilities[:, 1]

    print(f"✓ Added BERT Predicted Label ID")
    print(f"✓ Added BERT Predicted Label")
    print(f"✓ Added BERT Probability Class 0")
    print(f"✓ Added BERT Probability Class 1")
    print(f"\nFinal shape: {results_df.shape}")
    print(f"\nPreview:\n{results_df.head(3)}\n")

    return results_df

In [39]:
def create_confusion_matrix_dataframe(cm, id_to_label):
    """Add readable actual and predicted labels to the confusion matrix."""

    return pd.DataFrame(
        cm,
        index=[f"Actual: {id_to_label[0]}", f"Actual: {id_to_label[1]}"],
        columns=[f"Predicted: {id_to_label[0]}", f"Predicted: {id_to_label[1]}"],
    )

### Validation + Test

In [40]:
def evaluate_and_save_bert_results(
    trainer,
    tokenized_dataset,
    original_df,
    label_id_column,
    id_to_label,
    split_name,
    seed,
    output_dir,
    model_name,
    csv_prefix="bert_predictions",
):
    """Run full BERT prediction and evaluation pipeline and save results to disk."""

    print("\n" + "=" * 40)
    print(f"BERT EVALUATE AND SAVE: {split_name.upper()}")
    print("=" * 40)

    # ============================================================
    # CREATE SPLIT OUTPUT DIRECTORY
    # ============================================================
    split_dir = os.path.join(output_dir, split_name)
    os.makedirs(split_dir, exist_ok=True)
    print(f"✓ Output directory: {split_dir}")

    # ============================================================
    # PREDICTIONS
    # ============================================================
    y_pred, probabilities = get_finetuned_bert_predictions(
        trainer=trainer,
        tokenized_dataset=tokenized_dataset,
    )

    # ============================================================
    # GROUND TRUTH
    # ============================================================
    y_true = original_df[label_id_column].to_numpy()

    print(f"\nNumber of ground-truth labels : {len(y_true)}")
    print(f"First 5 ground-truth labels   : {y_true[:5]}")

    # ============================================================
    # SAVE PREDICTIONS DATAFRAME (with readable labels + probabilities)
    # ============================================================
    predictions_df = create_bert_results_dataframe(
        source_df=original_df,
        y_pred=y_pred,
        probabilities=probabilities,
        id_to_label=id_to_label,
    )

    DataProcessing.save_to_file(
        predictions_df,
        path=split_dir,
        prefix=csv_prefix,
        save_file_type="csv",
        include_version=False,
    )
    print(f"✓ Saved predictions CSV to: {os.path.join(split_dir, f'{csv_prefix}.csv')}")

    # ============================================================
    # EVALUATE AND SAVE METRICS
    # ============================================================
    metrics_df, cm, eval_report = evaluate_bert_predictions(
        y_true=y_true,
        y_pred=y_pred,
        probabilities=probabilities,
        split_name=split_name,
        seed=seed,
        save_path=split_dir,
        model_name=model_name,
    )

    DataProcessing.save_to_file(
        metrics_df,
        path=split_dir,
        prefix="bert_metrics_summary",
        save_file_type="csv",
        include_version=False,
    )
    print(f"✓ Saved metrics summary to: {os.path.join(split_dir, 'bert_metrics_summary.csv')}")

    # ============================================================
    # SAVE READABLE CONFUSION MATRIX
    # ============================================================
    confusion_matrix_df = create_confusion_matrix_dataframe(
        cm=cm,
        id_to_label=id_to_label,
    )

    print(f"\nConfusion Matrix:\n{confusion_matrix_df}\n")

    DataProcessing.save_to_file(
        confusion_matrix_df,
        path=split_dir,
        prefix="bert_confusion_matrix",
        save_file_type="csv",
        include_version=False,
    )
    print(f"✓ Saved confusion matrix to: {os.path.join(split_dir, 'bert_confusion_matrix.csv')}")

    return metrics_df, cm, eval_report

In [41]:
val_metrics_df, val_cm, val_report = evaluate_and_save_bert_results(
    trainer=trainer,
    tokenized_dataset=tokenized_val_dataset,
    original_df=val_df,
    label_id_column=label_id_column,
    id_to_label=id_to_label,
    split_name="validation",
    seed=seed,
    output_dir=output_dir,
    model_name=finetuned_model_name,
    csv_prefix="bert_predictions_val",
)

test_metrics_df, test_cm, test_report = evaluate_and_save_bert_results(
    trainer=trainer,
    tokenized_dataset=tokenized_test_dataset,
    original_df=test_df,
    label_id_column=label_id_column,
    id_to_label=id_to_label,
    split_name="test",
    seed=seed,
    output_dir=output_dir,
    model_name=finetuned_model_name,
    csv_prefix="bert_predictions_test",
)


BERT EVALUATE AND SAVE: VALIDATION
✓ Output directory: ../data/classification_results/naacl_2026_submission/naacl_2026_submission_2026-09-25/seed300/in_domain/bert-large-cased/validation

BERT PREDICTIONS


/Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/.venv_predictions/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


/Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/.venv_predictions/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Number of BERT predictions : 8
First 5 predicted labels   : [0 0 0 0 0]
Probability matrix shape   : (8, 2)
First probability vector   : [0.6175927  0.38240737]

Number of ground-truth labels : 8
First 5 ground-truth labels   : [1 1 0 0 0]

CREATE BERT RESULTS DATAFRAME
✓ Added BERT Predicted Label ID
✓ Added BERT Predicted Label
✓ Added BERT Probability Class 0
✓ Added BERT Probability Class 1

Final shape: (8, 8)

Preview:
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

/Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/.venv_predictions/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/.venv_predictions/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/.venv_predictions/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833

✓ Saved predictions CSV to: ../data/classification_results/naacl_2026_submission/naacl_2026_submission_2026-09-25/seed300/in_domain/bert-large-cased/validation/bert_predictions_val.csv

EVALUATE BERT: VALIDATION
              precision    recall  f1-score   support

           0       0.50      1.00      0.67         4
           1       0.00      0.00      0.00         4

    accuracy                           0.50         8
   macro avg       0.25      0.50      0.33         8
weighted avg       0.25      0.50      0.33         8

Confusion Matrix:
[[4 0]
 [4 0]]

TN: 4, FP: 0, FN: 4, TP: 0

Saving PNG file to: ../data/classification_results/naacl_2026_submission/naacl_2026_submission_2026-09-25/seed300/in_domain/bert-large-cased/validation/visualizations/confusion_matrix_tolsa-bert.png
✓ Saved confusion matrix: confusion_matrix_tolsa-bert.png

ROC-AUC Score: 0.2500

PR-AUC Score: 0.4571


METRICS SUMMARY WITH SEED 300
   seed       model       split train_accuracy val_accuracy  test

/Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/.venv_predictions/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Number of BERT predictions : 8
First 5 predicted labels   : [0 0 0 0 0]
Probability matrix shape   : (8, 2)
First probability vector   : [0.61284393 0.3871561 ]

Number of ground-truth labels : 8
First 5 ground-truth labels   : [0 0 1 1 0]

CREATE BERT RESULTS DATAFRAME
✓ Added BERT Predicted Label ID
✓ Added BERT Predicted Label
✓ Added BERT Probability Class 0
✓ Added BERT Probability Class 1

Final shape: (8, 8)

Preview:
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

/Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/.venv_predictions/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/.venv_predictions/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/.venv_predictions/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833

## Experiment Log

In [42]:
def create_bert_experiment_log(
    experiment_name,
    output_dir,
    dataset,
    text_column,
    label_column,
    model_name,
    finetuned_model_name,
    seed,
    val_size,
    test_size,
    epochs,
    learning_rate,
    weight_decay,
    train_batch_size,
    eval_batch_size,
    max_length,
    logging_steps,
    train_df,
    val_df,
    test_df,
    label_to_id,
):
    """Save a readable record of the BERT experiment configuration."""

    log_lines = [
        "=" * 40,
        "BERT SENTENCE CLASSIFICATION EXPERIMENT LOG",
        "=" * 40,
        f"Timestamp:              {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        f"Experiment:             {experiment_name}",
        f"Seed:                   {seed}",
        "",
        "--- Data ---",
        f"Dataset:                {dataset}",
        f"Text Column:            {text_column}",
        f"Label Column:           {label_column}",
        "",
        "--- Splits ---",
        f"Val Size:               {val_size}",
        f"Test Size:              {test_size}",
        f"Train Size:             {len(train_df)}",
        f"Validation Size:        {len(val_df) if val_df is not None else 'N/A'}",
        f"Test Size:              {len(test_df) if test_df is not None else 'N/A'}",
        "",
        "--- BERT Model ---",
        f"Pretrained Checkpoint:  {model_name}",
        f"Finetuned Model Name:   {finetuned_model_name}",
        f"Num Labels:             {len(label_to_id)}",
        "",
        "--- Training Arguments ---",
        f"Epochs:                 {epochs}",
        f"Learning Rate:          {learning_rate}",
        f"Weight Decay:           {weight_decay}",
        f"Train Batch Size:       {train_batch_size}",
        f"Eval Batch Size:        {eval_batch_size}",
        f"Max Token Length:       {max_length}",
        f"Logging Steps:          {logging_steps}",
        f"FP16:                   {torch.cuda.is_available()}",
        "",
        "--- Label Mapping ---",
    ]

    for label, label_id in label_to_id.items():
        log_lines.append(f"  {label_id} -> {label}")

    log_lines += [
        "",
        "--- Output ---",
        f"Output Directory:       {output_dir}",
        "=" * 40,
    ]

    log_dir = os.path.join(output_dir, "experiment_log")
    os.makedirs(log_dir, exist_ok=True)

    log_path = os.path.join(log_dir, "experiment_log.txt")

    with open(log_path, "w", encoding="utf-8") as f:
        f.write("\n".join(log_lines))

    print(f"✓ Experiment log saved to: {log_path}")

In [43]:
create_bert_experiment_log(
    experiment_name=experiment_name,
    output_dir=output_dir,
    dataset=dataset,
    text_column=text_column,
    label_column=label_column,
    model_name=model_name,
    finetuned_model_name=finetuned_model_name,
    seed=seed,
    val_size=val_size,
    test_size=test_size,
    epochs=epochs,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    train_batch_size=train_batch_size,
    eval_batch_size=eval_batch_size,
    max_length=max_length,
    logging_steps=logging_steps,
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    label_to_id=label_to_id,
)

print("\n" + "=" * 40)
print("PIPELINE COMPLETE")
print("=" * 40)
print(f"Experiment:          {experiment_name}")
print(f"Pretrained model:    {model_name}")
print(f"Finetuned model:     {finetuned_model_name}")
print(f"Seed:                {seed}")
print(f"\n✓ All outputs saved to: {output_dir}\n")

✓ Experiment log saved to: ../data/classification_results/naacl_2026_submission/naacl_2026_submission_2026-09-25/seed300/in_domain/bert-large-cased/experiment_log/experiment_log.txt

PIPELINE COMPLETE
Experiment:          naacl_2026_submission_2026-09-25
Pretrained model:    bert-large-cased
Finetuned model:     tolsa-bert
Seed:                300

✓ All outputs saved to: ../data/classification_results/naacl_2026_submission/naacl_2026_submission_2026-09-25/seed300/in_domain/bert-large-cased

